# Age Regression with a CNN

This notebook walks through the project step by step. The actual code lives in `src/age_estimation/`, the notebook only calls it and shows the results.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch

from age_estimation.config import DEFAULT_WEIGHTS, REPORTS_DIR, TrainConfig
from age_estimation.data import build_loader, download_dataset, load_file_names, split_data
from age_estimation.model import AgeCNN, get_device
from age_estimation.predict import predict_age
from age_estimation.train import evaluate, predict_all, train

device = get_device()
print(torch.__version__, device)

## 1. Data

Download UTKFace (only the first time) and read the age labels from the file names.

In [ ]:
image_dir = download_dataset()
files, labels = load_file_names(image_dir)
print(f"{len(files)} images, ages {min(labels)} to {max(labels)}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
for ax, i in zip(axes, np.random.default_rng(0).choice(len(files), 5, replace=False)):
    ax.imshow(plt.imread(files[i]))
    ax.set_title(f"Age: {labels[i]}")
    ax.axis("off")
plt.show()

Distribution of the labels. The data set is not balanced: there are many young adults and babies and few very old people.

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.hist(labels, bins=range(0, 120, 2))
plt.xlabel("Age")
plt.ylabel("Number of images")
plt.title("Age distribution in UTKFace")
plt.show()

## 2. Train / validation / test split

In [ ]:
config = TrainConfig(epochs=20)
torch.manual_seed(config.seed)

splits = split_data(files, labels, config.val_fraction, config.test_fraction, config.seed)
train_loader = build_loader(splits["train"], config.batch_size, shuffle=True, num_workers=config.num_workers)
val_loader = build_loader(splits["val"], config.batch_size, num_workers=config.num_workers)
test_loader = build_loader(splits["test"], config.batch_size, num_workers=config.num_workers)
{name: len(pairs) for name, pairs in splits.items()}

## 3. Model

In [ ]:
model = AgeCNN().to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Training

The loss is the mean absolute error, i.e. by how many years the prediction is off on average.

In [ ]:
history = train(model, train_loader, val_loader, config, device)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history["train_loss"], label="Training")
plt.plot(epochs, history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MAE (years)")
plt.title("Loss per epoch")
plt.legend()
plt.show()

## 5. Evaluation on the test set

As a reference: a model that always predicts the median age of the training data.

In [ ]:
preds, true = predict_all(model, test_loader, device)
test_mae = (preds - true).abs().mean().item()

train_ages = np.array([age for _, age in splits["train"]])
baseline_mae = np.abs(true.numpy() - np.median(train_ages)).mean()

print(f"Test MAE:     {test_mae:.2f} years")
print(f"Baseline MAE: {baseline_mae:.2f} years (always predicting the median age)")

In [ ]:
# Predicted vs. true age
plt.figure(figsize=(5, 5))
plt.scatter(true, preds, s=4, alpha=0.3)
plt.plot([0, 110], [0, 110], color="black", linewidth=1)
plt.xlabel("True age")
plt.ylabel("Predicted age")
plt.title("Test set")
plt.show()

## 6. Save the model and predict single images

In [ ]:
DEFAULT_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), DEFAULT_WEIGHTS)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / "history.json").write_text(json.dumps({**history, "test_loss": test_mae}, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
for ax, (path, age) in zip(axes, splits["test"][:5]):
    ax.imshow(plt.imread(path))
    ax.set_title(f"True: {age}  Predicted: {predict_age(model, path):.0f}")
    ax.axis("off")
plt.show()

To try your own picture (a square portrait with the face in the centre works best):

```python
predict_age(model, "path/to/your_image.jpg")
```